<a href="https://colab.research.google.com/github/vvelvadapu9/DemoAIProj/blob/Dev/Hyperparameter_Tuning_ANN_Optuna_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.4/247.4 kB 22.6 MB/s eta 0:00:00


In [ ]:
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import numpy as np

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x = np.concatenate((x_train, x_test), axis=0).astype("float32") / 255.0
y = np.concatenate((y_train, y_test), axis=0)

x = x.reshape(-1, 28 * 28)  # Flatten
x_train, x_val, y_train, y_val = train_test_split(x,
                                                  y,
                                                  test_size=0.2,
                                                  random_state=42)
print(x_train.shape)
print(y_train.shape)
print(x_val.shape)
print(y_val.shape)

In [ ]:
def objective(trial):
    # Hyperparameters
    hidden_units = trial.suggest_int("hidden_units", 64, 512)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2,log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["adam", "sgd"])

    # Model definition
    model = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(hidden_units, activation="relu"),
        layers.Dropout(dropout_rate),
        layers.Dense(10, activation="softmax")
    ])

    if optimizer_name == "sgd":
      optimizer = getattr(keras.optimizers, optimizer_name.upper())(learning_rate=learning_rate)
    else:
      optimizer = getattr(keras.optimizers, optimizer_name.title())(learning_rate=learning_rate)


    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    # Training
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        batch_size=128,
        epochs=5,
        verbose=0
    )

    # Return best validation accuracy
    return max(history.history["val_accuracy"])

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best trial:")
trial = study.best_trial
print(f"  Accuracy: {trial.value}")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2025-08-29 06:34:44,881] A new study created in memory with name: no-name-0b2e2d3f-ba43-471b-8631-1b068c8daaaf
[I 2025-08-29 06:35:09,301] Trial 0 finished with value: 0.9612143039703369 and parameters: {'hidden_units': 298, 'dropout_rate': 0.17728220824286708, 'learning_rate': 0.0002755453798111658, 'optimizer': 'adam'}. Best is trial 0 with value: 0.9612143039703369.
[I 2025-08-29 06:35:43,408] Trial 1 finished with value: 0.46385714411735535 and parameters: {'hidden_units': 463, 'dropout_rate': 0.15625330843167, 'learning_rate': 0.00013980184944766138, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.9612143039703369.
[I 2025-08-29 06:36:16,019] Trial 2 finished with value: 0.9738571643829346 and parameters: {'hidden_units': 466, 'dropout_rate': 0.20130778824669956, 'learning_rate': 0.006753116885119591, 'optimizer': 'adam'}. Best is trial 2 with value: 0.9738571643829346.
[I 2025-08-29 06:36:33,786] Trial 3 finished with value: 0.8696428537368774 and parameters: {'hidden_units

Best trial:
  Accuracy: 0.9787856936454773
    hidden_units: 424
    dropout_rate: 0.2781909028666678
    learning_rate: 0.0018067552026995956
    optimizer: adam


In [ ]:
optuna.visualization.plot_param_importances(study).show()

# **Tuning With Multiple Hidden Layers**

In [ ]:
def objective(trial):
    # Hyperparameters to tune
    n_layers = trial.suggest_int("n_layers", 1, 3)  # Number of hidden layers
    hidden_units = trial.suggest_int("hidden_units", 64, 512)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam","SGD"])

    # Build model dynamically
    model = keras.Sequential()
    model.add(layers.Input(shape=(784,)))   # input layer

    for i in range(n_layers):
        model.add(layers.Dense(hidden_units, activation="relu", name=f"hidden_{i+1}"))
        model.add(layers.Dropout(dropout_rate, name=f"dropout_{i+1}"))

    model.add(layers.Dense(10, activation="softmax", name="output"))  # output layer

    # Compile
    optimizer = getattr(keras.optimizers, optimizer_name)(learning_rate=learning_rate)
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    # Train
    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        batch_size=128,
        epochs=5,
        verbose=0
    )
    return max(history.history["val_accuracy"])

In [ ]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print("Best trial:")
trial = study.best_trial
print(f"  Accuracy: {trial.value}")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

[I 2025-08-29 07:43:29,615] A new study created in memory with name: no-name-d5c480d8-cc3f-484a-9c84-5d93b8915c38
[I 2025-08-29 07:44:01,177] Trial 0 finished with value: 0.8423571586608887 and parameters: {'n_layers': 2, 'hidden_units': 202, 'dropout_rate': 0.26548137265563404, 'learning_rate': 0.002503727691745832, 'optimizer': 'SGD'}. Best is trial 0 with value: 0.8423571586608887.
[I 2025-08-29 07:44:43,300] Trial 1 finished with value: 0.9691428542137146 and parameters: {'n_layers': 2, 'hidden_units': 405, 'dropout_rate': 0.444630489594831, 'learning_rate': 0.004691617961534065, 'optimizer': 'Adam'}. Best is trial 1 with value: 0.9691428542137146.
[I 2025-08-29 07:44:57,075] Trial 2 finished with value: 0.8604999780654907 and parameters: {'n_layers': 1, 'hidden_units': 97, 'dropout_rate': 0.4322138310873057, 'learning_rate': 0.003051536899601864, 'optimizer': 'SGD'}. Best is trial 1 with value: 0.9691428542137146.
[I 2025-08-29 07:45:34,869] Trial 3 finished with value: 0.97542858

KeyboardInterrupt: 

In [ ]:
optuna.visualization.plot_param_importances(study).show()

# **Tuning GenAI Model**

In [ ]:
import keras
import keras_nlp
import tensorflow as tf

# 1. Load tokenizer and model (Gemma 2B instruct preset)
MODEL_NAME = "gemma_instruct_2b_en"
tokenizer = keras_nlp.models.GemmaTokenizer.from_preset(MODEL_NAME)
lm = keras_nlp.models.GemmaCausalLM.from_preset(MODEL_NAME)

# 2. Example dataset (tiny toy dataset)
examples = [
    {"instruction": "Translate to French", "input": "Hello world", "output": "Bonjour le monde"},
    {"instruction": "Summarize", "input": "AI is transforming education.", "output": "AI reshapes learning"},
]

MAX_LEN = 128

def preprocess(example):
    prompt = f"Instruction: {example['instruction']}\nInput: {example['input']}\nResponse:"
    full_text = prompt + " " + example['output']
    tokens = tokenizer(full_text, max_length=MAX_LEN, truncation=True)
    return {"input_ids": tokens["input_ids"], "attention_mask": tokens["attention_mask"]}

train_ds = tf.data.Dataset.from_generator(
    lambda: (preprocess(e) for e in examples),
    output_signature={
        "input_ids": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "attention_mask": tf.TensorSpec(shape=(None,), dtype=tf.int32),
    },
).padded_batch(2, padded_shapes={"input_ids": [MAX_LEN], "attention_mask": [MAX_LEN]})

# 3. Attach LoRA adapters (parameter-efficient fine-tuning)
keras_nlp.peft.attach_lora(
    lm,
    target_modules=["self_attention.query", "self_attention.value"],
    rank=8,
    alpha=16,
    dropout=0.05,
)

# 4. Compile and train
loss_fn = keras_nlp.losses.CausalLMAccuracyAndCrossEntropy(from_logits=True)
lm.compile(optimizer=keras.optimizers.AdamW(learning_rate=2e-4), loss=loss_fn)

lm.fit(train_ds, epochs=3)

# 5. Test generation
prompt = "Instruction: Translate to French\nInput: Good morning\nResponse:"
print(lm.generate(prompt, tokenizer=tokenizer, max_length=50))